In [ ]:
from chromadb.config import Settings
import chromadb

from utils.var import content

client = chromadb.PersistentClient(path= content["vectorDBPath"])

In [ ]:
collection = client.get_collection(
    name="job_roles_DB"
)
collection

In [ ]:
res = collection.get(
    where={"roleID" : "computer_vision_engineer"},
    include=["embeddings" , "documents"]
)
res

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from utils.var import content
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

def get_CV_embeddings(CVPath):
    loader = PyPDFLoader(CVPath)
    CV = loader.load()

    text = ""
    for i in range(len(CV)):
      text += CV[i].page_content
    
    textSplitter = RecursiveCharacterTextSplitter(
                chunk_size=500,
                chunk_overlap=50,
                separators=["\n\n", "\n", ".", " ", ""]
            )
    textSplitted = textSplitter.split_text(text)

    embeddingModel = SentenceTransformer("all-MiniLM-L6-v2")
    vectors = embeddingModel.encode(
                textSplitted,
                batch_size=16,
                normalize_embeddings=True,
                show_progress_bar=True
            ).tolist()
    
    return vectors

    

   


In [ ]:
def text_extractor(filePath):
    loader = PyPDFLoader(filePath)
    CV = loader.load()

    text = ""
    for i in range(len(CV)):
      text += CV[i].page_content

    return text

In [ ]:
Vectors = get_CV_embeddings(content["exCV"])
Vectors

In [ ]:
text = text_extractor(content["exCV"])

In [ ]:
from openai import OpenAI

OpenAIClient = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=content["HFTOKEN"],
)

def getRole(CVText):

    try:

        prompt_roles = f'''You are an ATS classifier.

Your task:
- Identify the closest matching job role from the allowed list.
- If no match exists, return "false".

Allowed roles:
computer_vision_engineer
nlp_engineer
devops_engineer
web_developer
fullstack_engineer
frontend_engineer
backend_engineer
data_analyst
data_scientist
ai_engineer
ml_engineer

Return ONLY one role name or "false"..
        '''

        messages=[
            {"role": "system", "content": prompt_roles},
            {"role": "user", "content": CVText}
        ]
        completion = OpenAIClient.chat.completions.create(
        model="openai/gpt-oss-20b:groq",
        messages = messages,
        temperature=0
            )

        role = completion.choices[0].message.content.strip()
        return role
    
    except Exception as e:
        print(e)

In [ ]:
role = getRole(text)
role

In [ ]:
type(role)

In [ ]:
type(Vectors)

In [ ]:
def loadRoleVectors(collectionName:str , directoryPath:str , role:str):
    try:

        client = chromadb.PersistentClient(path= directoryPath)
        collection = client.get_collection(
        name=collectionName
        )

        roleVectors  = collection.get(
        where={"roleID" : role},
        include=["embeddings" , "documents"]
        )

        return roleVectors

    except Exception as e:
        print(e)

roleVectors = loadRoleVectors(collectionName="job_roles_DB" , directoryPath=content["vectorDBPath"] , role=role)
roleVectors
    
    

In [ ]:
def similaritySearch_LLM(CVText , RoleText , role):
    try:
        SystemPrompt = '''you are automating CVs for comapny. Your job is to Rank provided CV text with requirements of that job. 
        return only score in range of 1 to 100, '''
        UserPrompt = f'''
        text of CV:{CVText} ,,,, 
        text of requriements for {role}:{RoleText}'''

        messages = [
            {"role":"system" , "content":SystemPrompt},
            {"role":"user" , "content":UserPrompt}
        ]

        completion = OpenAIClient.chat.completions.create(
        model="openai/gpt-oss-20b:groq",
        messages = messages,
        temperature=0
            )
        
        score = completion.choices[0].message.content.strip()
        return score
        
        
    except Exception as e:
        print(e)

In [ ]:
score = similaritySearch_LLM(text , roleVectors['documents'] , role)

In [ ]:
score

In [ ]:
import numpy as np

print(res['embeddings'].shape)
CVvectors = np.array(Vectors)
print(CVvectors.shape)

In [ ]:
RoleEmbeddings = res['embeddings']

RoleEmbeddings_centroid = RoleEmbeddings.mean(axis=0)
CVvectors_centroid = CVvectors.mean(axis=0)

cos_sim = np.dot(RoleEmbeddings_centroid, CVvectors_centroid) / (
    np.linalg.norm(CVvectors_centroid) * np.linalg.norm(CVvectors_centroid)
)

cos_sim

In [ ]:
matrixMul = RoleEmbeddings @ CVvectors.T
matrixMul

In [ ]:
print(np.mean(matrixMul , axis=1))
print(np.mean(matrixMul , axis=0))
print(matrixMul.max())


In [ ]:
from sklearn.metrics.pairwise import euclidean_distances

dist = 1 - ((euclidean_distances(RoleEmbeddings, CVvectors)) / 2)
print(max(dist))
print(min(dist))
